# XGBoost as feature selector

In [4]:
import os
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

## Get data

In [13]:
fold_path = "/home/alecacciatore/ECML26/GNN4eGFR/"
file_path = os.path.join(fold_path, "XY_temp.csv")
out_path = os.path.join(fold_path, "features_importance_score", "xgb_scores")
os.makedirs(out_path, exist_ok=True)

# read the CSV file into a DataFrame
df = pd.read_csv(file_path)

# the last column is the target variable and the first one is an index
X = df.iloc[:, 1:-3] # TODO:: include general practitioner features?
y_classif = df.iloc[:, -1]
y_regress = df.iloc[:, -2]

# print per-class distribution
five_class_counts = y_classif.value_counts().sort_index()
print("Class distribution:")
print(y_classif.value_counts())

# y_classif contains [I, II, IIIa, IIIb, IV, V] labels
# convert them to numerical labels for classification
label_mapping = {'I': 0, 'II': 1, 'IIIa': 1, 'IIIb': 1, 'IV': 1, 'V': 1}
y_classif = y_classif.map(label_mapping)

Class distribution:
eGFR_class
II      4925
I       1650
IIIa    1565
IIIb     720
IV       260
V         45
Name: count, dtype: int64


In [15]:
# Samples per class before SMOTE
print("Samples per class before SMOTE:")
print(y_classif.value_counts())

# Compute imbalance ratio
value_counts = y_classif.value_counts()
imbalance_ratio = value_counts.max() / value_counts.min()
print(f"Imbalance ratio: {imbalance_ratio}")

smote = SMOTE(random_state=42)
X, y_classif = smote.fit_resample(X, y_classif)

# New imbalance ratio
value_counts_resampled = y_classif.value_counts()
imbalance_ratio_resampled = value_counts_resampled.max() / value_counts_resampled.min()
print(f"Imbalance ratio after SMOTE: {imbalance_ratio_resampled}")

# save imbalance ratios
with open(os.path.join(out_path, "imbalance_ratio.txt"), "w") as f:
    f.write(f"5-class samples per class before SMOTE: {five_class_counts.to_dict()}\n")
    f.write(f"2-class samples per class before SMOTE: {value_counts}\n")
    f.write(f"Original imbalance ratio: {imbalance_ratio}\n")
    f.write(f"2-class samples per class after SMOTE: {value_counts_resampled}\n")
    f.write(f"Imbalance ratio after SMOTE: {imbalance_ratio_resampled}\n")

Samples per class before SMOTE:
eGFR_class
1    7515
0    7515
Name: count, dtype: int64
Imbalance ratio: 1.0
Imbalance ratio after SMOTE: 1.0


## Classification with XGBoost

In [18]:
# train XGBoost classifier with weighted cross-entropy loss
classifier = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)

parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
}

model_classif = GridSearchCV(estimator=classifier,
                             param_grid=parameters,
                             scoring='accuracy',
                             n_jobs=-1,
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                             verbose=1)
model_classif.fit(X, y_classif)

# save best parameters model
best_params_classif = model_classif.best_params_
with open(os.path.join(out_path, "xgb_classifier_best_params.txt"), "w") as f:
    for param, value in best_params_classif.items():
        f.write(f"{param}: {value}\n")

# compute classification accuracy on test set
best_model_classif = model_classif.best_estimator_
accuracy = best_model_classif.score(X, y_classif)
print(f"Classification test accuracy: {accuracy}")
with open(os.path.join(out_path, "xgb_classifier_accuracy.txt"), "w") as f:
    f.write(f"Test accuracy: {accuracy}\n")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Classification test accuracy: 0.9793080505655356


## Feature importance extraction

In [19]:
best_model = model_classif.best_estimator_
best_model.fit(X, y_classif)

# save feature importances
feature_importances = best_model.feature_importances_
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df.to_csv(os.path.join(out_path, "xgb_feature_importances.csv"), index=False)

In [20]:
# get 50 most important features 
top_50_features = importance_df.head(50)
top_50_features.to_csv(os.path.join(out_path, "xgb_top_50_features.csv"), index=False)

## Re-train with top features

In [24]:
# create a new dataset with only the top 50 features
feature_importances = best_model.feature_importances_

# undersample X to only top 50 features
top_50_feature_names = top_50_features['Feature'].tolist()
X_top_50 = X[top_50_feature_names]

# New classifier training with top 50 features
classifier_top_50 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)
parameters_top_50 = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
}

model_classif_top_50 = GridSearchCV(estimator=classifier_top_50,
                             param_grid=parameters_top_50,
                             scoring='accuracy',
                             n_jobs=-1,
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                             verbose=1)
model_classif_top_50.fit(X_top_50, y_classif)

# save best parameters model
best_params_classif_top_50 = model_classif_top_50.best_params_
with open(os.path.join(out_path, "xgb_classifier_top_50_best_params.txt"), "w") as f:
    for param, value in best_params_classif_top_50.items():
        f.write(f"{param}: {value}\n")

# compute classification accuracy on test set with top 50 features
best_model_classif_top_50 = model_classif_top_50.best_estimator_
best_model_classif_top_50.fit(X_top_50, y_classif)
accuracy_top_50 = best_model_classif_top_50.score(X_top_50, y_classif)
print(f"Classification test accuracy with top 50 features: {accuracy_top_50}")
with open(os.path.join(out_path, "xgb_classifier_top_50_accuracy.txt"), "w") as f:
    f.write(f"Test accuracy with top 50 features: {accuracy_top_50}\n")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Classification test accuracy with top 50 features: 0.9384564204923487


## Re-train with random 50 features

In [25]:
# Re-train with random 50 features
import random
all_features = X.columns.tolist()
random_50_features = random.sample(all_features, 50)
X_random_50 = X[random_50_features]

# New classifier training with random 50 features
classifier_random_50 = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)
parameters_random_50 = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
}

model_classif_random_50 = GridSearchCV(estimator=classifier_random_50,
                             param_grid=parameters_random_50,
                             scoring='accuracy',
                             n_jobs=-1,
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                             verbose=1)
model_classif_random_50.fit(X_random_50, y_classif)

# save best parameters model
best_params_classif_random_50 = model_classif_random_50.best_params_
with open(os.path.join(out_path, "xgb_classifier_random_50_best_params.txt"), "w") as f:
    for param, value in best_params_classif_random_50.items():
        f.write(f"{param}: {value}\n")

# compute classification accuracy on test set with random 50 features
best_model_classif_random_50 = model_classif_random_50.best_estimator_
best_model_classif_random_50.fit(X_random_50, y_classif)
accuracy_random_50 = best_model_classif_random_50.score(X_random_50, y_classif)
print(f"Classification test accuracy with random 50 features: {accuracy_random_50}")
with open(os.path.join(out_path, "xgb_classifier_random_50_accuracy.txt"), "w") as f:
    f.write(f"Test accuracy with random 50 features: {accuracy_random_50}\n")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Classification test accuracy with random 50 features: 0.844045242847638
